# Algoritmo de Dijkstra

## Caminhos Mínimos em Grafos Valorados com Arestas Não-Negativas

Autor: Prof. Lucas Nunes Alegre

In [ ]:
from typing import Dict, List, Tuple, Optional
import heapq
import random
import time

In [ ]:
# Grafos Valorados
# Dicionário mapeando cada nodo para uma lista de pares (vizinho, peso da aresta)
# Não pode haver arestas negativas!

G1 = {
    'S': [('A', 1), ('B', 3)],
    'A': [('S', 1), ('D', 5), ('C', 4)],
    'B': [('S', 3), ('D', 4), ('C', 1)],
    'C': [('B', 1), ('A', 4), ('E', 6)],
    'D': [('A', 5), ('B', 4), ('E', 2)],
    'E': [('D', 2), ('C', 6)],
    'F': [],
}

In [ ]:
def dijkstra_naive(graph: Dict, source: str) -> Tuple[Dict, Dict]:
    """
    Dijkstra O(|V||E|) sem fila de prioridades.

    Parâmetros:
      graph: dicionário (lista de adjacência) representando grafo valorado com arestas não-negativas.
      source: nó de origem.

    Retorna:
      dist: dicionário com a menor distância da origem para cada nó.
            Para nós inalcançáveis, distância = float('inf').
      parent: dicionário com o predecessor imediato no caminho mínimo
              (None para a origem e para nós inalcançáveis).
    """

    # Inicialização
    processado = {v: False for v in graph.keys()}
    dist = {v: float('inf') for v in graph.keys()}
    parent = {v: None for v in graph.keys()}

    dist[source] = 0  # Origem começa com distância zero

    # Lista de arestas (u, v, peso)
    arestas = []
    for u, vizinhos in graph.items():
        for v in vizinhos:
            arestas.append((u, v[0], v[1]))

    # Laço principal
    num_processados = 0
    while num_processados < len(graph):
        # Escolhe o nó não processado com menor distância
        u = None
        menor_dist = float('inf')
        for v in graph.keys():
            if not processado[v] and dist[v] < menor_dist:
                menor_dist = dist[v]
                u = v
        if u is None:
            break         # Se não encontrou nenhum alcançável, termina

        processado[u] = True
        num_processados += 1

        # Relaxamento das arestas saindo de u
        for (origem, destino, peso) in arestas:
            if origem == u and not processado[destino]:
                if dist[u] + peso < dist[destino]:
                    dist[destino] = dist[u] + peso
                    parent[destino] = u

    return dist, parent

In [ ]:
def reconstruct_path(parent: Dict, dist: Dict, target: str) -> List:
    """
    Reconstrói o menor caminho até o target usando o dicionário parent.
    Retorna lista vazia se inalcançável.
    """
    # Unreachable (and not the source) if distance is inf
    if dist.get(target, float('inf')) == float('inf'):
        return []

    path = []
    cur = target
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    path.reverse()
    return path

In [ ]:
dist, parent = dijkstra_naive(G1, 'S')
print(dist)
reconstruct_path(parent, dist, 'E')

In [ ]:
def dijkstra(graph: Dict, source: str) -> Tuple[Dict, Dict]:
    """
    Dijkstra O((|E| + |V|) log |V|) com heap binário.

    Parâmetros:
      graph: dicionário (lista de adjacência) representando grafo valorado com arestas não-negativas.
      source: nó de origem.

    Retorna:
      dist: dicionário com a menor distância da origem para cada nó.
            Para nós inalcançáveis, distância = float('inf').
      parent: dicionário com o predecessor imediato no caminho mínimo
              (None para a origem e para nós inalcançáveis).
    """

    # Inicialização
    processado = set()
    dist = {v: float('inf') for v in graph.keys()}
    parent = {v: None for v in graph.keys()}

    dist[source] = 0  # Origem começa com distância zero

    fila = [(0.0, source)]

    while fila:  # enquanto fila não for vazia
        du, u = heapq.heappop(fila)  # extrai nodo com menor distância na fila
    
        if u in processado:
            continue
        processado.add(u)

        for v, wv in graph[u]:    # para cada vizinho v de u
            novo_dv = du + wv     # distância até u mais peso da aresta u-v
            if novo_dv < dist[v]: # se distância menor que a atual
                dist[v] = novo_dv
                parent[v] = u
                heapq.heappush(fila, (novo_dv, v))  # coloca v na fila com sua distância atual da origem

    return dist, parent

In [ ]:
dist, parent = dijkstra(G1, 'S')
print(dist)
reconstruct_path(parent, dist, 'E')

## Benchmark

In [ ]:
def generate_random_graph(n: int,
                          avg_degree: int = 6,
                          weight_low: int = 1,
                          weight_high: int = 10,
                          directed: bool = False,
                          seed: Optional[int] = None) -> Dict:
    """
    Generate a random graph with strictly positive weights using the same adjacency format
    used in the examples. All nodes appear as keys. Optionally ensures connectivity by
    first adding a random spanning tree, then sprinkling extra edges.

    Parameters
    ----------
    n : int
        Number of nodes (v0, v1, ..., v{n-1}).
    avg_degree : int, optional
        Desired average out-degree per node (approximate).
    weight_low, weight_high : int, optional
        Inclusive range for integer positive weights.
    directed : bool, optional
        If False, add symmetric edges (undirected as two directed arcs).
    seed : Optional[int], optional
        Random seed for reproducibility.

    Returns
    -------
    Adj
        Adjacency dict: node -> list of (neighbor, positive_weight).
    """
    assert n >= 1, "n must be >= 1"
    if seed is not None:
        random.seed(seed)

    nodes = [f'v{i}' for i in range(n)]
    graph = {u: [] for u in nodes}

    # Helper to add an edge if not duplicate and not self-loop
    def add_edge(u: str, v: str, w: int):
        if u == v:
            return
        # prevent duplicate (u,v)
        if not any(x == v for (x, _) in graph[u]):
            graph[u].append((v, float(w)))

    # 1) Build a random spanning tree to ensure connectivity (n-1 edges)
    for i in range(1, n):
        u = nodes[i]
        v = nodes[random.randrange(0, i)]  # connect to a previous node
        w = random.randint(weight_low, weight_high)
        add_edge(u, v, w)
        if not directed:
            add_edge(v, u, w)

    # 2) Sprinkle additional edges to reach target density
    # total target edges (outgoing count) ~ n * avg_degree
    current_edges = sum(len(adj) for adj in graph.values())
    target_edges = max(n - 1, n * avg_degree)
    attempts = 0
    max_attempts = 10 * (target_edges - current_edges + 1)

    while current_edges < target_edges and attempts < max_attempts:
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u != v:
            w = random.randint(weight_low, weight_high)
            before = len(graph[u])
            add_edge(u, v, w)
            after = len(graph[u])
            if after > before:
                current_edges += 1
                if not directed:
                    # Add symmetric edge if it didn't exist
                    before2 = len(graph[v])
                    add_edge(v, u, w)
                    if len(graph[v]) > before2:
                        current_edges += 1
        attempts += 1

    return graph

In [ ]:
for n in [100, 1_000, 10_000, 100_000]: #, 1_000_000]:
    # === Benchmark on a random positive-weight graph ===
    avg_deg = 6     # average out-degree
    seed = 42       # for reproducibility
    directed = True

    G = generate_random_graph(n, avg_degree=avg_deg, weight_low=1, weight_high=10, directed=directed, seed=seed)
    src = 'v0'

    # Time heap-based Dijkstra
    t0 = time.perf_counter()
    dist_heap, parent_heap = dijkstra(G, src)
    t1 = time.perf_counter()

    print(f"\n=== Timing (n={n}, avg_deg={avg_deg}, directed={directed}) ===")
    print(f"Heap Dijkstra:   {t1 - t0:.6f} s")

    # Time naive Dijkstra
    t0 = time.perf_counter()
    dist_heap, parent_heap = dijkstra_naive(G, src)
    t1 = time.perf_counter()

    print(f"\n=== Timing (n={n}, avg_deg={avg_deg}, directed={directed}) ===")
    print(f"Naive Dijkstra:   {t1 - t0:.6f} s")